# Studio 2 — Kompetisi Klasifikasi Sampah: Reproduksi Run Terbaik (`putaran2_ck224`)

Notebook ini mereproduksi **run terbaik** kelompok 11: ConvNeXt-Tiny dengan resep resmi torchvision
(`Resize(236)` lalu `CenterCrop(224)`), augmentasi `balanced`, pemanasan kepala 3 epoch, dan
penyempurnaan seluruh layer 10 epoch. Hasil run ini pada split validasi beku 413 gambar adalah
**macro-F1 0,9753** dan akurasi **0,9782** (9 gambar salah), dan modelnya adalah yang aktif di
`submission/`.

Notebook ini bisa dijalankan di Google Colab maupun di laptop/VS Code. Nilai konfigurasi di bawah
sengaja identik dengan perintah CLI yang dipakai saat run itu dijalankan, sehingga hasilnya dapat
dibandingkan langsung.

## 0. Alur notebook

1. Konfigurasi percobaan (sel 1) dan pemeriksaan runtime (sel 2).
2. Penyiapan repositori dan lokasi dataset (sel 3 sampai 4).
3. Pemeriksaan split validasi beku, tanpa menulis apa pun (sel 5).
4. Pelatihan (sel 6), evaluasi resmi (sel 7), grafik (sel 8).
5. Pemeriksaan ukuran bobot dan fp16 (sel 9).
6. Promosi ke `submission/` mengikuti ambang yang sudah tercatat (sel 10).
7. Perintah commit dan catatan kendala (sel 11 sampai 13).

Pelatihan penuh berjalan sekitar 18 menit pada GPU 4 GB (RTX 2050) dan sekitar 10 menit pada
GPU Colab T4.

## 1. Konfigurasi

Semua nilai di bawah adalah nilai yang benar-benar dipakai pada run terbaik. `LIMIT` hanya untuk
uji cepat pipeline: nilainya **tidak boleh** dipakai untuk menilai skor, karena daftar data latih
diurutkan per kelas sehingga 24 gambar pertama seluruhnya berasal dari kelas pertama.

In [ ]:
import os
import random
import sys
from pathlib import Path

EXP_NAME     = 'putaran2_ck224'
CANDIDATE    = 'experiments/convnext_ck224.py'
EPOCHS       = 10          # tahap 2; tahap 1 (kepala) 3 epoch dari berkas model
BATCH_SIZE   = 16
LR           = 0.0001
WD           = 0.05
AUG          = 'balanced'
PATIENCE     = 0
SEED         = 42
WORKERS      = 2
HEAD_EPOCHS  = None        # None berarti mengikuti HEAD_EPOCHS pada berkas model
LIMIT        = 0           # 0 = seluruh 2340 gambar latih; >0 hanya untuk uji cepat

# Ambang promosi: rekor kelompok sendiri pada split beku 413 gambar.
SKOR_TERCATAT = {'putaran2_ck224 (crop 224, resize 236)': 0.9753}
PERCOBAAN_INI  = 'putaran2_ck224'

# Angka pembanding tim lain, hanya untuk ditampilkan, tidak dipakai memutuskan promosi.
PEMBANDING = {'peringkat 1 putaran lalu (diukur ulang pada split sendiri)': 0.9754}

REPO_URL = 'https://github.com/SigitKurnia05/DL-RA_studio2-klasifikasi-sampah.git'
BRANCH   = 'main'
NAMA_REPO = REPO_URL.rsplit('/', 1)[-1].replace('.git', '')

# Satu konstanta yang boleh diedit per lokasi. Di Colab nilainya folder hasil clone;
# di laptop nilainya folder repositori yang sudah ada.
KANDIDAT_REPO = [
    Path('/content') / NAMA_REPO,
    Path(r'D:\Perkuliahan duniawi\Semester 7\Pembelajaran Mendalam\Tugas studio 2\DL-RA_studio2-klasifikasi-sampah'),
    Path.cwd(),
    Path.cwd().parent,
]
REPO_DIR = next((p for p in KANDIDAT_REPO if (p / 'train.py').is_file() and (p / 'experiments').is_dir()), None)
if REPO_DIR is None:
    raise SystemExit('Repositori tidak ditemukan. Set REPO_DIR secara manual pada sel ini.')

# Folder kerja run ini. Di laptop mengikuti konvensi repo (runs/<tag>/), di Colab ke /content
# supaya tidak menumpuk di dalam repositori hasil clone.
if Path('/content').exists():
    WORK_DIR = Path('/content') / f'kerja_{EXP_NAME}'
else:
    WORK_DIR = REPO_DIR / 'runs' / EXP_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)

OUT_WEIGHTS = WORK_DIR / 'model.pth'
HIST_OUT    = WORK_DIR / 'history.json'
EVAL_OUT    = WORK_DIR / 'metrics.json'
PLOT_OUT    = WORK_DIR / 'curves.png'
TRAIN_LOG   = WORK_DIR / 'train.log'

# Folder yang menjadi CWD saat pelatihan; train.py menulis runs/best.pth dan my_val/ relatif ke sini.
RUMAH_KERJA = REPO_DIR

jam = {'EXP_NAME': EXP_NAME, 'CANDIDATE': CANDIDATE, 'EPOCHS': EPOCHS, 'BATCH_SIZE': BATCH_SIZE,
       'LR': LR, 'WD': WD, 'AUG': AUG, 'PATIENCE': PATIENCE, 'SEED': SEED, 'HEAD_EPOCHS': HEAD_EPOCHS}
for k, v in jam.items():
    print(f'{k:12s}: {v}')
print(f'{"REPO_DIR":12s}: {REPO_DIR}')
print(f'{"WORK_DIR":12s}: {WORK_DIR}')

## 2. Pemeriksaan runtime

Memastikan Python, torch, dan GPU yang dipakai. Pada Colab, paket inti (torch, torchvision,
matplotlib, numpy) sudah tersedia; tidak ada pemasangan paket di dalam notebook.

In [ ]:
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
try:
    import numpy as np
    np.random.seed(SEED)
except ImportError:
    np = None
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
except ImportError:
    torch = None

print('python       :', sys.version.split()[0])
print('executable   :', sys.executable)
if torch is not None:
    print('torch        :', torch.__version__)
    import torchvision
    print('torchvision  :', torchvision.__version__)
    print('cuda tersedia:', torch.cuda.is_available())
    if torch.cuda.is_available():
        nama = torch.cuda.get_device_name(0)
        total = torch.cuda.get_device_properties(0).total_memory / 2 ** 30
        print('gpu          :', nama, f'({total:.1f} GiB)')
        if total < 8:
            print('catatan: VRAM di bawah 8 GiB, pastikan BATCH_SIZE tidak melebihi 16 pada crop 224.')
else:
    raise SystemExit('torch tidak tersedia pada kernel ini. Pilih kernel yang memiliki torch.')

## 3. Repositori dan Google Drive

Di Colab, Drive hanya di-mount bila folder dataset yang dituju memang berada di dalamnya, dan
repositori di-clone (atau ditarik bila sudah ada). Di laptop, sel ini hanya melaporkan repositori
dan commit yang dipakai, tanpa menyentuh apa pun dari jaringan.

In [ ]:
import subprocess

DI_COLAB = Path('/content').exists()

if DI_COLAB:
    import shutil
    tujuan_repo = Path('/content') / NAMA_REPO
    if not (tujuan_repo / '.git').is_dir():
        print('clone repositori...')
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(tujuan_repo)], check=True)
    else:
        print('repositori sudah ada, tarik pembaruan...')
        subprocess.run(['git', '-C', str(tujuan_repo), 'pull', 'origin', BRANCH], check=True)
    REPO_DIR = tujuan_repo

    # Mount Drive hanya bila folder datanya memang di Drive dan belum terpasang.
    drive_terpasang = Path('/content/drive/MyDrive').exists()
    if not drive_terpasang:
        from google.colab import drive
        drive.mount('/content/drive')
    print('google drive : terpasang' if Path('/content/drive/MyDrive').exists() else 'google drive : tidak terpasang')

os.chdir(RUMAH_KERJA)
print('cwd          :', Path.cwd())
print('repositori   :', REPO_DIR)
commit = subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'],
                        capture_output=True, text=True).stdout.strip()
print('commit       :', commit)
kotor = subprocess.run(['git', '-C', str(REPO_DIR), 'status', '--short'],
                       capture_output=True, text=True).stdout.strip()
print('status git   :', 'bersih' if not kotor else 'ADA PERUBAHAN (periksa .gitignore)')

## 4. Penyiapan dataset

Dataset tidak di-commit (aturan kompetisi). Yang dibutuhkan hanya folder berisi lima subfolder
kelas. Nilai `DATA_DIR` menyesuaikan lokasi: di Colab menunjuk Drive, di laptop menunjuk
`data/train` di dalam repositori.

In [ ]:
KELAS = ['botol_plastik', 'gelas_plastik', 'kertas_kardus', 'saset_kemasan', 'styrofoam']

KANDIDAT_DATA = [
    Path('/content/drive/MyDrive/Studio 2/train'),
    Path('/content/drive/MyDrive/Studio 2/data/train'),
    REPO_DIR / 'data' / 'train',
    REPO_DIR / 'train',
]
DATA_DIR = next((p for p in KANDIDAT_DATA if p.is_dir() and all((p / k).is_dir() for k in KELAS)), None)
if DATA_DIR is None:
    dicari = '\n  '.join(str(p) for p in KANDIDAT_DATA)
    raise SystemExit('Folder dataset tidak ditemukan. Periksa lokasi berikut:\n  ' + dicari)

print('dataset       :', DATA_DIR)
total = 0
for k in KELAS:
    n = len(list((DATA_DIR / k).glob('*.jpg')))
    total += n
    print(f'  {k:16s} {n:5d} gambar')
print('total         :', total, 'gambar')
if total < 2700:
    raise SystemExit('Jumlah gambar jauh di bawah 2753; periksa apakah dataset sudah lengkap.')

# Validasi folder untuk evaluate.py; train.py menyalin gambar validasi ke sini bila belum ada.
MY_VAL_DIR = RUMAH_KERJA / 'my_val'
print('folder uji    :', MY_VAL_DIR, '(dibuat otomatis oleh train.py bila belum ada)')

## 5. Pemeriksaan split validasi beku

Split validasi tidak pernah dibuat ulang di dalam notebook ini. Sel di bawah hanya memeriksa
kecocokannya dengan data dan melaporkan hash kanoniknya.

In [ ]:
r = subprocess.run([sys.executable, str(REPO_DIR / 'make_split.py'), '--check',
                    '--data', str(DATA_DIR),
                    '--out', str(REPO_DIR / 'splits' / 'val.txt')],
                   cwd=str(RUMAH_KERJA), capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())
if r.returncode != 0:
    raise SystemExit('Pemeriksaan split gagal; jangan lanjutkan pelatihan.')

import hashlib
isi = (REPO_DIR / 'splits' / 'val.txt').read_bytes()
akar = hashlib.sha256(isi).hexdigest()
print('splits/val.txt:', len([b for b in isi.decode('utf-8').splitlines() if b.strip()]), 'baris')
print('sha256        :', akar[:16], '...')
if akar[:16] != '36bee95ec932553b':
    print('PERINGATAN: hash berbeda dari hash kanonik 36bee95ec932553b; periksa garis akhir berkas (CRLF/LF).')

## 6. Pelatihan

Perintah di bawah identik dengan run terbaik. Output pelatihan tampil langsung di sel ini dan
sekaligus disimpan ke `train.log`. Setiap epoch mencetak macro-F1 validasi; bobot terbaik dipilih
berdasarkan macro-F1, bukan akurasi.

In [ ]:
cmd = [
    sys.executable, str(REPO_DIR / 'train.py'),
    '--data',        str(DATA_DIR),
    '--split',       str(REPO_DIR / 'splits' / 'val.txt'),
    '--model',       str(REPO_DIR / CANDIDATE),
    '--out',         str(OUT_WEIGHTS),
    '--history-out', str(HIST_OUT),
    '--epochs',      str(EPOCHS),
    '--bs',          str(BATCH_SIZE),
    '--lr',          str(LR),
    '--wd',          str(WD),
    '--aug',         AUG,
    '--patience',    str(PATIENCE),
    '--workers',     str(WORKERS),
    '--seed',        str(SEED),
]
if HEAD_EPOCHS is not None:
    cmd += ['--head-epochs', str(HEAD_EPOCHS)]
if LIMIT:
    cmd += ['--limit', str(LIMIT)]

print('perintah:'); print(' '.join(cmd)); print()
with open(TRAIN_LOG, 'w', encoding='utf-8') as f:
    proses = subprocess.Popen(cmd, cwd=str(RUMAH_KERJA), stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
    for baris in proses.stdout:
        print(baris, end='')
        f.write(baris)
    kode_keluar = proses.wait()
print('\nkode keluar:', kode_keluar)
if kode_keluar != 0:
    raise SystemExit('Pelatihan gagal; lihat keluaran di atas dan berkas ' + str(TRAIN_LOG))

## 7. Evaluasi resmi

Skrip `evaluate.py` milik asisten dijalankan apa adanya, pada pasangan `model.py` + `model.pth` di
folder kerja. Loader-nya memakai `strict=True`, sehingga pasangan yang tidak cocok akan langsung
gagal di sini, bukan nanti di lingkungan penilai.

In [ ]:
import shutil
shutil.copyfile(REPO_DIR / CANDIDATE, WORK_DIR / 'model.py')

# evaluate.py menuntut my_val/ berisi lima subfolder kelas. Bila belum ada (folder baru di Colab,
# atau laptop yang belum pernah menjalankan pelatihan), folder itu disiapkan dari split beku.
if not all((MY_VAL_DIR / k).is_dir() and any((MY_VAL_DIR / k).iterdir()) for k in KELAS):
    print('my_val/ belum lengkap; menyalin gambar validasi dari split beku...')
    r = subprocess.run([sys.executable, str(REPO_DIR / 'make_split.py'),
                        '--materialize', str(MY_VAL_DIR),
                        '--data', str(DATA_DIR),
                        '--out', str(REPO_DIR / 'splits' / 'val.txt')],
                       cwd=str(RUMAH_KERJA), capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
    if r.returncode != 0:
        raise SystemExit('Gagal menyiapkan folder my_val/.')

r = subprocess.run([sys.executable, str(REPO_DIR / 'evaluate.py'),
                    '--submission', str(WORK_DIR), '--data', str(MY_VAL_DIR),
                    '--out', str(EVAL_OUT)],
                   cwd=str(RUMAH_KERJA), capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())
if r.returncode != 0:
    raise SystemExit('Evaluasi gagal.')

import json
met = json.loads(EVAL_OUT.read_text(encoding='utf-8'))
hist = json.loads(HIST_OUT.read_text(encoding='utf-8'))
puncak = max(hist, key=lambda h: h['val_macro_f1'])

def angka(nilai, desimal=4):
    return f'{nilai:.{desimal}f}'.replace('.', ',')

print()
print('macro-F1 (evaluate.py) :', angka(met['macro_f1']))
print('akurasi  (evaluate.py) :', angka(met['accuracy']))
print('macro-F1 (riwayat)     :', angka(puncak['val_macro_f1']), 'pada epoch', puncak['epoch'])
print('jumlah gambar salah    :', met['n_images'] - round(met['accuracy'] * met['n_images']))
for k, v in met['per_class_f1'].items():
    print(f'  F1 {k:16s} {angka(v)}')

## 8. Grafik dan ringkasan per epoch

Tiga panel: kurva loss latih dan validasi, macro-F1 dan akurasi per epoch, serta confusion matrix
dari evaluasi resmi. Gambar disimpan ke `curves.png` sebelum ditampilkan.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

ep  = [h['epoch'] for h in hist]
tr  = [h['train_loss'] for h in hist]
va  = [h['val_loss'] for h in hist]
f1  = [h['val_macro_f1'] for h in hist]
acc = [h['val_acc'] for h in hist]
puncak_i = max(range(len(f1)), key=lambda i: f1[i])

print('%3s %10s %9s %10s %8s  %s' % ('Ep', 'TrainLoss', 'ValLoss', 'MacroF1', 'Acc', 'Rekor'))
print('-' * 56)
for h in hist:
    tanda = 'BEST' if h['epoch'] == ep[puncak_i] else ''
    print('%3d %10.4f %9.4f %10.4f %8.4f  %s' % (h['epoch'], h['train_loss'], h['val_loss'],
                                                h['val_macro_f1'], h['val_acc'], tanda))
print()
print('rata-rata waktu per epoch: %.1f detik | total %.1f menit'
      % (sum(h['detik'] for h in hist) / len(hist), sum(h['detik'] for h in hist) / 60))

fig, ax = plt.subplots(1, 3, figsize=(19, 5.2))
ax[0].plot(ep, tr, 'o-', label='latih')
ax[0].plot(ep, va, 's-', label='validasi')
ax[0].set_title('Loss per epoch'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('cross-entropy')
ax[0].legend(); ax[0].grid(alpha=0.3); ax[0].xaxis.set_major_locator(MaxNLocator(integer=True))

ax[1].plot(ep, f1, 'o-', color='#2ca02c', label='macro-F1')
ax[1].plot(ep, acc, '^--', color='#7f7f7f', label='akurasi')
ax[1].axhline(0.92, color='#d62728', ls=':', lw=1.5, label='baseline asisten sekitar 0,92')
ax[1].axhline(0.9753, color='#1f77b4', ls=':', lw=1.5, label='rekor kelompok 0,9753')
ax[1].scatter([ep[puncak_i]], [f1[puncak_i]], s=180, marker='*', color='#ff7f0e', zorder=5,
              label='terbaik %.4f (ep %d)' % (f1[puncak_i], ep[puncak_i]))
ax[1].set_title('Macro-F1 dan akurasi'); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('skor')
ax[1].legend(fontsize=9); ax[1].grid(alpha=0.3); ax[1].xaxis.set_major_locator(MaxNLocator(integer=True))

cm = met['confusion_matrix']
label = [k[:6] for k in KELAS]
ax[2].imshow(cm, cmap='Blues')
batas = max(max(b) for b in cm) / 2
for i in range(len(cm)):
    for j in range(len(cm)):
        ax[2].text(j, i, cm[i][j], ha='center', va='center',
                   color='white' if cm[i][j] > batas else 'black')
ax[2].set_xticks(range(len(label))); ax[2].set_xticklabels(label, rotation=30, ha='right')
ax[2].set_yticks(range(len(label))); ax[2].set_yticklabels(label)
ax[2].set_title('Confusion matrix (macro-F1 %.4f)' % met['macro_f1'])
ax[2].set_xlabel('prediksi'); ax[2].set_ylabel('label asli')

fig.suptitle('%s - %s, aug %s, bs %d, lr %s, wd %s' % (EXP_NAME, CANDIDATE, AUG, BATCH_SIZE, LR, WD))
fig.tight_layout()
fig.savefig(PLOT_OUT, dpi=150, bbox_inches='tight')
print('grafik disimpan:', PLOT_OUT)
plt.show()

## 9. Pemeriksaan ukuran bobot dan fp16

GitHub menolak berkas di atas 100 MB. Sejak `train.py` versi terbaru, bobot terbaik sudah disimpan
dalam fp16 sehingga berukuran sekitar 53 MB; sel ini memverifikasi hal itu, dan hanya mengonversi
bila ternyata masih fp32.

In [ ]:
import torch

ukuran = OUT_WEIGHTS.stat().st_size
print('ukuran keluaran: %.1f MB (%s byte)' % (ukuran / 2 ** 20, format(ukuran, ',')))
print('batas keras GitHub: 100 MB')

if ukuran > 100 * 2 ** 20:
    print('melewati batas; dikonversi ke fp16...')
    shutil.copyfile(OUT_WEIGHTS, WORK_DIR / 'model_fp32.pth')
    sd = torch.load(OUT_WEIGHTS, map_location='cpu', weights_only=True)
    bukan_float = [k for k, v in sd.items() if not v.is_floating_point()]
    if bukan_float:
        raise SystemExit('ada tensor non-float sehingga fp16 tidak aman: ' + str(bukan_float[:3]))
    torch.save({k: v.half() for k, v in sd.items()}, OUT_WEIGHTS)
    print('ukuran setelah fp16: %.1f MB' % (OUT_WEIGHTS.stat().st_size / 2 ** 20))
else:
    sd = torch.load(OUT_WEIGHTS, map_location='cpu', weights_only=True)
    ragam = sorted({str(v.dtype) for v in sd.values()})
    print('dtype tensor   :', ', '.join(ragam))
    print('jumlah tensor  :', len(sd))
    print('parameter      :', format(sum(v.numel() for v in sd.values()), ','))

# Bukti bahwa pasangan model.py + model.pth tetap dapat dimuat penilai.
r = subprocess.run([sys.executable, str(REPO_DIR / 'evaluate.py'),
                    '--submission', str(WORK_DIR), '--data', str(MY_VAL_DIR)],
                   cwd=str(RUMAH_KERJA), capture_output=True, text=True)
print(r.stdout.strip().splitlines()[0] if r.stdout.strip() else r.stderr.strip())
if r.returncode != 0:
    raise SystemExit('Pasangan yang tersimpan gagal dimuat ulang oleh evaluate.py.')
print('load_state_dict(strict=True) melalui evaluate.py: lolos')

## 10. Promosi ke `submission/`

Promosi hanya dilakukan bila skor percobaan ini melebihi rekor kelompok sendiri, dan bila selisihnya
di bawah 0,01 akan diberi peringatan karena satu gambar pada split 413 gambar bernilai sekitar 0,0024.

In [ ]:
skor = max(h['val_macro_f1'] for h in hist)
terbaik = max(SKOR_TERCATAT.values())
dipromosikan = skor > terbaik

print('skor percobaan ini          : %.4f (%s)' % (skor, PERCOBAAN_INI))
print('ambang (rekor kelompok)     : %.4f' % terbaik)
print('selisih                     : %+.4f' % (skor - terbaik))
for nama, nilai in PEMBANDING.items():
    print('pembanding (tidak dipakai)  : %.4f (%s)' % (nilai, nama))
print()

if dipromosikan:
    if skor - terbaik < 0.01:
        print('[PERINGATAN] selisih di bawah 0,01; satu gambar pada split ini bernilai sekitar 0,0024,')
        print('            jadi kemenangannya belum dapat dipastikan berbeda secara statistik.')
    shutil.copyfile(REPO_DIR / CANDIDATE, REPO_DIR / 'submission' / 'model.py')
    shutil.copyfile(OUT_WEIGHTS, REPO_DIR / 'submission' / 'model.pth')
    print('dipromosikan: submission/model.py dan submission/model.pth diperbarui.')
else:
    print('tidak dipromosikan: skor tidak melebihi %.4f, submission/ dibiarkan apa adanya.' % terbaik)

## 11. Perintah commit dan push

Notebook ini tidak melakukan commit otomatis. Perintah berikut dijalankan manual dari root
repositori, dan `submission/model.py` wajib sepasang dengan `submission/model.pth` dalam satu commit.

In [ ]:
def jalur_untuk_commit(p):
    """Jalur relatif terhadap repositori bila memungkinkan; jalur absolut bila di luar (mis. di Colab)."""
    try:
        return p.relative_to(REPO_DIR).as_posix()
    except ValueError:
        return str(p)

print('Perintah commit untuk run ini (jalankan manual dari root repositori):')
print()
print('  git add submission/model.py submission/model.pth')
print('  git add %s' % ' '.join(jalur_untuk_commit(x) for x in (HIST_OUT, EVAL_OUT, PLOT_OUT)))
print('  git commit -m "feat(submission): ConvNeXt-Tiny %s (macro-F1 %s)"' % (EXP_NAME, angka(skor)))
print('  git -c credential.helper=wincred push origin main')
print()
print('Berkas yang sengaja tidak di-commit: data/, my_val/, seluruh berkas .pth kecuali')
print('submission/model.pth, serta runs/**/model.py dan runs/last.json (lihat .gitignore).')

## 12. Catatan kendala yang sudah terukur

Semua angka di bawah diukur pada split beku 413 gambar dengan `evaluate.py` resmi, dan dicatat supaya
percobaan berikutnya tidak mengulanginya:

| Konfigurasi berbeda | Hasil | Catatan |
|---|---|---|
| crop 224 / resize 236 (resep run ini) | **0,9753** | 9 gambar salah |
| crop 224, `WEIGHT_POWER` 1,25, 14 epoch | 0,9752 | recall kertas_kardus tetap 49/55 |
| crop 224, augmentasi `standard` | 0,9708 | kardus naik ke 50/55, kelas lain turun |
| crop 256 / resize 288 | 0,9717 | kardus terbaik sepanjang malam (52/55, F1 0,9630) |
| crop 320 / resize 448 dari ImageNet | 0,9596 | 16 gambar salah |
| crop 320, fine-tune dari bobot 224 | 0,9593 | 14 gambar salah |
| bias logit kardus (21 nilai grid) | <= 0,9753 | presisi kardus sudah 100 persen |
| rata-rata bobot antar percobaan | <= 0,9705 | menyatukan tidak berhasil |
| sapuan rasio uji (20 konfigurasi) | <= 0,9753 | memperbaiki 1 gambar, merusak 1 gambar lain |

Kesimpulan yang berlaku sampai sekarang: batas kertas_kardus versus kelas lain bersifat
tukar-menukar. Setiap konfigurasi yang memperbaiki gambar kardus sasaran membayarnya dengan dua
sampai sembilan gambar lain, sehingga 0,9753 masih merupakan hasil terbaik.

## 13. Tabel hasil antar-percobaan

| Percobaan | Backbone | Crop | Augmentasi | LR / WD | Epoch | Macro-F1 | Akurasi |
|---|---|---|---|---|---|---|---|
| eksperimen 3 | EfficientNetV2-S | 288 | strong | 1,5e-4 / 1e-2 | 11 | 0,9595 | 0,9613 |
| eksperimen 4 | ConvNeXt-Tiny | 224 | strong | 1e-4 / 1e-2 | 9 | 0,9658 | 0,9685 |
| eksperimen 5 | ConvNeXt-Tiny | 224 (resize 236) | balanced | 1e-4 / 5e-2 | 13 | **0,9753** | **0,9782** |
| eksperimen 6 | ConvNeXt-Tiny | 256 (resize 288) | balanced | 1e-4 / 5e-2 | 13 | 0,9717 | 0,9709 |

Kolom macro-F1 dan akurasi adalah nilai pada split validasi lokal kelompok sendiri (413 gambar,
seed 42), bukan skor pada validation set tersembunyi. Model aktif di `submission/` adalah
eksperimen 5 (`putaran2_ck224`).